In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot  as plt
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


### Load The Dataset

In [2]:
df_2009_2010 = pd.read_excel("online_retail_II.xlsx",sheet_name='Year 2009-2010')

In [3]:
df_2010_2011 = pd.read_excel("online_retail_II.xlsx",sheet_name='Year 2010-2011')

In [5]:
df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)


### Make a Copy()

In [6]:
data = df.copy()

### Standardise Column Names

In [7]:
data.columns = (
    data.columns
        .str.strip()
        .str.replace(" ", "_")
)

In [8]:
data.columns.tolist()

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer_ID',
 'Country']

### Data Types

In [9]:
data["InvoiceDate"] = pd.to_datetime(
    data["InvoiceDate"],
    errors="coerce"
)

In [11]:
data["Customer_ID"] = pd.to_numeric(
    data["Customer_ID"],
    errors="coerce"
).astype("Int64")

### Investigate Duplicates

In [12]:
duplicates = data[
    data.duplicated(keep=False)
]

print("Duplicate rows:", len(duplicates))

duplicates.head(20)

Duplicate rows: 67242


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom


In [13]:
data.duplicated().sum()

np.int64(34335)

In [14]:
data = data.drop_duplicates().copy()

In [15]:
print("Shape after duplicate removal:", data.shape)
print("Remaining duplicates:", data.duplicated().sum())

Shape after duplicate removal: (1033036, 8)
Remaining duplicates: 0


In [17]:
data = data.dropna(subset=["Customer_ID"])

In [19]:
customer_data = data[
    data["Customer_ID"].notna()
].copy()

In [20]:
print("Master transaction data:", data.shape)
print("Customer analysis data:", customer_data.shape)

Master transaction data: (797885, 8)
Customer analysis data: (797885, 8)


### Description recovery

In [21]:
missing_desc = data[
    data["Description"].isna()
]

print(
    "Missing descriptions:",
    len(missing_desc)
)

Missing descriptions: 0


In [22]:
description_map = (
    data.dropna(subset=["Description"])
        .drop_duplicates(subset=["StockCode"])
        .set_index("StockCode")["Description"]
)

In [23]:
data["Description"] = (
    data["Description"]
        .fillna(
            data["StockCode"].map(description_map)
        )
)

In [24]:
print(
    "Remaining missing descriptions:",
    data["Description"].isna().sum()
)

Remaining missing descriptions: 0


### Investigate unrecoverable descriptions

In [28]:
unresolved_description = data[
    data["Description"].isna()
]

unresolved_description[
    [
        "StockCode",
        "Invoice",
        "Quantity",
        "Price",
        "Customer_ID"
    ]
].head(50)

,StockCode,Invoice,Quantity,Price,Customer_ID


### Transaction type

In [30]:
data["TransactionType"] = np.where(
    data["Invoice"]
        .astype(str)
        .str.startswith("C"),
    "Cancelled",
    "Sale"
)

In [31]:
data["TransactionType"].value_counts()

TransactionType
Sale         779495
Cancelled     18390
Name: count, dtype: int64

### Investigate negative quantities

In [32]:
negative_qty = data[
    data["Quantity"] < 0
].copy()

In [34]:
negative_qty["IsCancellationInvoice"] = (
    negative_qty["Invoice"]
        .astype(str)
        .str.startswith("C")
)

In [35]:
negative_qty[
    "IsCancellationInvoice"
].value_counts()

IsCancellationInvoice
True    18390
Name: count, dtype: int64

In [36]:
pd.crosstab(
    negative_qty["IsCancellationInvoice"],
    columns="Count"
)

col_0,Count
IsCancellationInvoice,
True,18390


### Investigate non-C negative quantities

In [37]:
negative_non_cancelled = negative_qty[
    ~negative_qty["IsCancellationInvoice"]
]

print(
    "Negative quantity with non-C invoice:",
    len(negative_non_cancelled)
)

Negative quantity with non-C invoice: 0


In [39]:
negative_non_cancelled[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer_ID"
    ]
].head(50)

,Invoice,StockCode,Description,Quantity,Price,Customer_ID


### Investigate zero prices

In [41]:
zero_price = data[
    data["Price"] == 0
].copy()

print(
    "Zero-price records:",
    len(zero_price)
)

Zero-price records: 70


In [42]:
zero_price[
    "Description"
].value_counts(
    dropna=False
).head(30)

Description
Manual                               7
CHRISTMAS PUDDING TRINKET POT        2
This is a test product.              2
REGENCY CAKESTAND 3 TIER             2
DOOR MAT FAIRY CAKE                  1
CHRISTMAS CRAFT WHITE FAIRY          1
ANTIQUE LILY FAIRY LIGHTS            1
 FLAMINGO LIGHTS                     1
ANTIQUE GLASS HEART DECORATION       1
CHARLOTTE BAG , SUKI DESIGN          1
RETRO SPOT LARGE MILK JUG            1
VINTAGE GLASS COFFEE CADDY           1
6 RIBBONS EMPIRE                     1
CAST IRON HOOK GARDEN TROWEL         1
CAST IRON HOOK GARDEN FORK           1
HANGING METAL BIRD BATH              1
AIRLINE BAG VINTAGE JET SET WHITE    1
SET/5 RED SPOTTY LID GLASS BOWLS     1
DOORMAT HOME SWEET HOME BLUE         1
TV DINNER TRAY DOLLY GIRL            1
MILK PAN PINK RETROSPOT              1
POLYESTER FILLER PAD 45x45cm         1
CAKE STAND LACE WHITE                1
DOLLY GIRL LUNCH BOX                 1
NOEL WOODEN BLOCK LETTERS            1
RED RETROSPOT

In [43]:
zero_price[
    "StockCode"
].value_counts().head(30)

StockCode
M          7
22065      2
TEST001    2
22423      2
48185      1
22142      1
85042      1
79320      1
21143      1
22355      1
21533      1
21662      1
22076      1
22459      1
22458      1
21765      1
22376      1
20914      1
22690      1
22472      1
22202      1
46000M     1
22218      1
22630      1
22121      1
21843      1
22624      1
22846      1
22845      1
22841      1
Name: count, dtype: int64

In [45]:
zero_price[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Customer_ID"
    ]
].head(50)

,Invoice,StockCode,Description,Quantity,Customer_ID
4674,489825,22076,6 RIBBONS EMPIRE,12,16126
6781,489998,48185,DOOR MAT FAIRY CAKE,2,15658
16107,490727,M,Manual,1,17231
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,14108
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,14108
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,15070
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,18071
47126,493761,79320,FLAMINGO LIGHTS,24,14258
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,12417
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,16858


### Negative prices

In [47]:
negative_price = data[
    data["Price"] < 0
]

negative_price[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer_ID"
    ]
]

,Invoice,StockCode,Description,Quantity,Price,Customer_ID


### Create Revenue

In [49]:
data["Revenue"] = (
    data["Quantity"] *
    data["Price"]
)

In [50]:
data["Revenue"].describe()

count    797885.000000
mean         20.416465
std         313.518824
min     -168469.600000
25%           4.350000
50%          11.700000
75%          19.500000
max      168469.600000
Name: Revenue, dtype: float64

### Create date features

In [51]:
data["Year"] = data["InvoiceDate"].dt.year

data["Month"] = data["InvoiceDate"].dt.month

data["MonthName"] = data["InvoiceDate"].dt.month_name()

data["Day"] = data["InvoiceDate"].dt.day

data["DayName"] = data["InvoiceDate"].dt.day_name()

data["Week"] = data["InvoiceDate"].dt.isocalendar().week.astype(int)

data["Hour"] = data["InvoiceDate"].dt.hour

In [52]:
data[
    [
        "InvoiceDate",
        "Year",
        "Month",
        "MonthName",
        "Day",
        "DayName",
        "Week",
        "Hour"
    ]
].head()

,InvoiceDate,Year,Month,MonthName,Day,DayName,Week,Hour
0,2009-12-01 07:45:00,2009,12,December,1,Tuesday,49,7
1,2009-12-01 07:45:00,2009,12,December,1,Tuesday,49,7
2,2009-12-01 07:45:00,2009,12,December,1,Tuesday,49,7
3,2009-12-01 07:45:00,2009,12,December,1,Tuesday,49,7
4,2009-12-01 07:45:00,2009,12,December,1,Tuesday,49,7


### Create an analytical sales dataset

In [54]:
sales_data = data[
    (data["TransactionType"] == "Sale") &
    (data["Quantity"] > 0) &
    (data["Price"] > 0)
].copy()

In [55]:
print(
    "Sales dataset:",
    sales_data.shape
)

Sales dataset: (779425, 17)


### Customer analytical dataset

In [57]:
customer_data = sales_data[
    sales_data["Customer_ID"].notna()
].copy()

In [58]:
print(
    "Customer analytical dataset:",
    customer_data.shape
)

Customer analytical dataset: (779425, 17)


### Product demand dataset

In [59]:
demand_data = sales_data[
    [
        "InvoiceDate",
        "StockCode",
        "Description",
        "Quantity",
        "Revenue"
    ]
].copy()

In [60]:
demand_data.head()

,InvoiceDate,StockCode,Description,Quantity,Revenue
0,2009-12-01 07:45:00,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,83.4
1,2009-12-01 07:45:00,79323P,PINK CHERRY LIGHTS,12,81.0
2,2009-12-01 07:45:00,79323W,WHITE CHERRY LIGHTS,12,81.0
3,2009-12-01 07:45:00,22041,"RECORD FRAME 7"" SINGLE SIZE",48,100.8
4,2009-12-01 07:45:00,21232,STRAWBERRY CERAMIC TRINKET BOX,24,30.0


### Save datasets

In [63]:
from pathlib import Path

BASE_DIR = Path("..")

RAW_DIR = BASE_DIR / "data" / "raw"
CLEANED_DIR = BASE_DIR / "data" / "cleaned"

CLEANED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Cleaned folder:", CLEANED_DIR)
print("Folder ready:", CLEANED_DIR.exists())

Cleaned folder: ..\data\cleaned
Folder ready: True


In [65]:
data.to_csv(
    CLEANED_DIR / "retail_master_cleaned.csv",
    index=False
)

print("Master dataset saved successfully.")

Master dataset saved successfully.


In [66]:
sales_data.to_csv(
    CLEANED_DIR / "retail_sales.csv",
    index=False
)

customer_data.to_csv(
    CLEANED_DIR / "retail_customer_analysis.csv",
    index=False
)

demand_data.to_csv(
    CLEANED_DIR / "retail_demand.csv",
    index=False
)

print("All datasets saved successfully.")

All datasets saved successfully.


In [67]:
from pathlib import Path

CLEANED_DIR = Path("../data/cleaned")

files = [
    "retail_master_cleaned.csv",
    "retail_sales.csv",
    "retail_customer_analysis.csv",
    "retail_demand.csv"
]

for file in files:
    path = CLEANED_DIR / file
    print(f"{file}: {'✓ Found' if path.exists() else '✗ Missing'}")

retail_master_cleaned.csv: ✓ Found
retail_sales.csv: ✓ Found
retail_customer_analysis.csv: ✓ Found
retail_demand.csv: ✓ Found
